# AI Assignment - does the statement follow?

**Name:** Ashutosh Ravi Gupta  
**Roll no:** P25CS0002

Context is a few English sentences (facts and if-then rules). I have to say True or False for a statement.

English -> pattern parser -> facts + rules -> forward chaining -> True / False

No model decides the answer. The parser is just regexes. The reasoner keeps firing rules until nothing new shows up, then looks for the query. Negation is closed-world (if I never derived it, the negation counts as true). That matches RuleTaker. There is a flag to turn it off, and I compare both later.

There is a small check after the representation, after the parser, and after the reasoner, so I can see which part broke.

In [5]:
import re, time, csv, os, random
from collections import defaultdict
import pandas as pd

try:
    import datasets
except ImportError:
    !pip -q install datasets
    import datasets

## 1. Representation

A literal is `(negated, predicate, args)`. `young(bob)` has one argument, `chase(cat, dog)` has two. A variable is a string starting with `?`, so `?x` stands for "someone" / "they".

A rule is a list of body literals and one head. "big and round" is stored as two rules with the same body.

In [6]:
# ---------- representation ----------
# a literal is a tuple (negated, predicate, args) ; args are strings,
# variables start with '?'
def lit(pred, *args, neg=False):
    return (neg, pred, tuple(args))

def is_var(t):
    return isinstance(t, str) and t.startswith('?')

def show_lit(l):
    neg, p, args = l
    s = p + '(' + ', '.join(args) + ')'
    return ('not ' if neg else '') + s

class Rule:
    def __init__(self, body, head, text=''):
        self.body = body      # list of literals
        self.head = head      # single literal
        self.text = text
    def __repr__(self):
        return ' & '.join(show_lit(b) for b in self.body) + ' -> ' + show_lit(self.head)

In [7]:
# make sure the tuples are what I think they are before the parser builds on them
assert lit('young', 'bob') == (False, 'young', ('bob',))
assert lit('cold', 'dog', neg=True) == (True, 'cold', ('dog',))
assert is_var('?x') and not is_var('bob') and not is_var(3)
assert show_lit(lit('chase', 'cat', 'dog')) == 'chase(cat, dog)'
assert show_lit(lit('cold', 'dog', neg=True)) == 'not cold(dog)'
r = Rule([lit('young', '?x'), lit('kind', '?x')], lit('successful', '?x'))
assert repr(r) == 'young(?x) & kind(?x) -> successful(?x)'
print('representation checks passed')

representation checks passed


## 2. Parser

The sentences I care about are pretty repetitive, so this is regexes, not a real grammar.

Covered: `Bob is young`, `The cat does not like the dog`, `Bob is red and kind`, `If someone is young then they are smart`, `Bob is big if Bob is red`, `All rabbits are furry`, `Kind, young things are not smart`.

`chases` and `chase` have to land on the same predicate, so I strip a trailing s. Anything I cannot read is kept in a failed list. FOLIO is normal English, so a lot of it misses. If the query itself does not parse I just check whether that sentence is already written in the context.

In [8]:
# ---------- parser ----------
QUANT_WORDS = {'someone', 'something', 'somebody', 'anyone', 'anything', 'anybody',
               'everyone', 'everything', 'everybody', 'people', 'things'}
PRONOUNS = {'they', 'it', 'he', 'she', 'them', 'him', 'her', 'itself'}
VERB_STOP = {'is', 'are', 'does', 'do', 'has', 'have', 'was', 'were'}

def clean(s):
    s = s.strip().lower()
    s = re.sub(r'[.!?]+$', '', s)
    s = s.replace("doesn't", "does not").replace("don't", "do not")
    s = s.replace("isn't", "is not").replace("aren't", "are not")
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def lemma(verb):
    # cheap lemmatiser, enough for the verbs that show up in these datasets
    if verb in ('is', 'are'): return 'is'
    if verb in ('does', 'do'): return 'do'
    if verb in ('has', 'have'): return 'have'
    if verb.endswith('ies') and len(verb) > 4: return verb[:-3] + 'y'
    if verb.endswith(('shes', 'ches', 'xes', 'zes')): return verb[:-2]
    if verb.endswith('s') and not verb.endswith('ss'): return verb[:-1]
    return verb

def parse_term(s):
    """entity phrase -> constant or variable"""
    s = s.strip()
    if s in QUANT_WORDS or s in PRONOUNS:
        return '?x'
    s = re.sub(r'^(the|a|an) ', '', s)
    if s in QUANT_WORDS or s in PRONOUNS:
        return '?x'
    return s.replace(' ', '_')

def parse_pred_phrase(subj, rest):
    """subj is already a term. rest is e.g. 'is not red' / 'likes the dog' / 'does not chase the cat'"""
    rest = rest.strip()
    m = re.match(r'^(is|are) not (?:a |an |the )?(.+)$', rest)
    if m: return lit(m.group(2).replace(' ', '_'), subj, neg=True)
    m = re.match(r'^(is|are) (?:a |an |the )?(.+)$', rest)
    if m: return lit(m.group(2).replace(' ', '_'), subj)
    m = re.match(r'^(does|do) not (\w+) (.+)$', rest)
    if m: return lit(lemma(m.group(2)), subj, parse_term(m.group(3)), neg=True)
    m = re.match(r'^(\w+) (.+)$', rest)
    if m and m.group(1) not in ('not', 'and', 'then', 'if'):
        return lit(lemma(m.group(1)), subj, parse_term(m.group(2)))
    return None

def split_subject(clause):
    """returns (subject_phrase, rest) or None"""
    toks = clause.split()
    if not toks: return None
    if toks[0] in ('all', 'every', 'any', 'no'):
        return None
    if toks[0] in ('the', 'a', 'an'):
        # entity can be 1..3 words; stop at the first verb-like token
        for n in range(1, 4):
            if len(toks) <= n + 1: break
            nxt = toks[n + 1]
            if nxt in VERB_STOP or (nxt.endswith('s') and not nxt.endswith('ss')):
                return ' '.join(toks[:n + 1]), ' '.join(toks[n + 1:])
        return None
    if len(toks) >= 2:
        return toks[0], ' '.join(toks[1:])
    return None

def anchor(lits):
    # "it/they" points at someone/something if the body has one.
    # otherwise it points at the last subject ("if the cat is big then it is red").
    for l in lits:
        for a in l[2]:
            if is_var(a):
                return a
    return lits[-1][2][0]

def parse_clause(clause, default_subj=None):
    # "bob is red and kind" -> red(bob), kind(bob)
    clause = clause.strip()
    parts = [p.strip() for p in re.split(r'\band\b', clause) if p.strip()]
    lits = []
    subj = default_subj
    quant = default_subj if is_var(default_subj) else None
    for p in parts:
        l = None
        sp = split_subject(p)
        if sp:
            s, rest = sp
            term = parse_term(s)
            if s in PRONOUNS:
                if quant is not None:
                    term = quant
                elif subj is not None:
                    term = subj
            elif is_var(term):
                quant = term
            l = parse_pred_phrase(term, rest)
            if l:
                subj = term
        if l is None and subj is not None:
            # leftover piece: "kind", "not rough"
            l = parse_pred_phrase(subj, p) or parse_pred_phrase(subj, 'is ' + p)
        if l is None:
            return None
        lits.append(l)
    return lits

NOUN_RE = r'(things?|people|persons?|animals?|ones?|someone|somebody)'

def parse_sentence(sent):
    """returns ('fact', [literals]) | ('rule', [Rule]) | None"""
    s = clean(sent)
    if not s: return None

    # if ... then ...
    m = re.match(r'^if (.+?),? then (.+)$', s)
    if m:
        body = parse_clause(m.group(1))
        if body is None: return None
        heads = parse_clause(m.group(2), default_subj=anchor(body))
        if heads is None: return None
        return ('rule', [Rule(body, h, sent) for h in heads])

    # ... if ...   (reversed conditional)
    m = re.match(r'^(.+?) if (.+)$', s)
    if m:
        body = parse_clause(m.group(2))
        if body:
            heads = parse_clause(m.group(1), default_subj=anchor(body))
            if heads:
                return ('rule', [Rule(body, h, sent) for h in heads])

    # 'all things that are red are big' / 'someone who is rough is quiet'
    m = re.match(r'^(?:all |every |any )?(?:' + NOUN_RE + r'|someone|something|anyone|anything|everyone|everything) (?:that|who|which) (.+?) (is|are|does not|do not|\w+s) (.+)$', s)
    if m:
        body = parse_clause('?x ' + m.group(2))
        heads = parse_clause('?x ' + m.group(3) + ' ' + m.group(4), default_subj='?x')
        if body and heads:
            return ('rule', [Rule(body, h, sent) for h in heads])

    # all red, kind things are big   /  red things like the dog
    m = re.match(r'^(?:all |every |any )?([a-z, ]+?) ' + NOUN_RE + r' (.+)$', s)
    if m and m.group(1) not in ('the', 'a', 'an', 'all', 'every', 'any'):
        adjs = [a.strip() for a in re.split(r',|\band\b', m.group(1)) if a.strip()]
        body = [lit(a.replace(' ', '_'), '?x') for a in adjs]
        heads = parse_clause('?x ' + m.group(3), default_subj='?x')
        if heads is None:
            heads = parse_clause('?x is ' + m.group(3), default_subj='?x')
        if heads:
            return ('rule', [Rule(body, h, sent) for h in heads])

    # 'all rabbits are furry' / 'every red cat likes the dog'
    m = re.match(r'^(?:all|every|any) ([a-z ]+?) (is|are|does not|do not|\w+s) (.+)$', s)
    if m:
        words = m.group(1).split()
        body = [lit(w, '?x') for w in words[:-1]] + [lit(lemma(words[-1]), '?x')]
        heads = parse_clause('?x ' + m.group(2) + ' ' + m.group(3), default_subj='?x')
        if heads:
            return ('rule', [Rule(body, h, sent) for h in heads])

    # plain fact(s)
    lits = parse_clause(s)
    if lits is None: return None
    if any(is_var(a) for l in lits for a in l[2]):
        # 'someone is red' on its own is not a usable fact
        return None
    return ('fact', lits)

def parse_context(context):
    facts, rules, failed = [], [], []
    sents = [x for x in re.split(r'(?<=[.!?])\s+', context.strip()) if x.strip()]
    for sent in sents:
        out = parse_sentence(sent)
        if out is None:
            failed.append(sent)
        elif out[0] == 'fact':
            facts.extend(out[1])
        else:
            rules.extend(out[1])
    return facts, rules, failed

In [9]:
# poke the parser with the shapes I actually rely on
cases = [
    ("Bob is young.", 'fact', 'young(bob)'),
    ("The cat does not like the dog.", 'fact', 'not like(cat, dog)'),
    ("Bob is red and kind.", 'fact', 'kind(bob)'),
    ("The bald eagle chases the cat.", 'fact', 'chase(bald_eagle, cat)'),
    ("If someone is young, then they are smart.", 'rule', 'young(?x) -> smart(?x)'),
    ("If someone is smart and kind, then they are successful.", 'rule', 'kind(?x) -> successful(?x)'),
    ("If someone is big and not rough then they are not green.", 'rule', 'not rough(?x)'),
    ("All rabbits are furry.", 'rule', 'rabbit(?x) -> furry(?x)'),
    ("Kind, young things are not smart.", 'rule', 'young(?x) -> not smart(?x)'),
    ("Someone who is rough is quiet.", 'rule', 'rough(?x) -> quiet(?x)'),
    ("If the cat is big then it is red.", 'rule', 'big(cat) -> red(cat)'),
    ("Bob is big if Bob is red.", 'rule', 'red(bob) -> big(bob)'),
    # "it" is the someone, not the bear that showed up later in the body
    ("If something eats the bear and the bear eats the rabbit then it likes the cat.", 'rule', 'like(?x, cat)'),
]
for sent, kind, expect in cases:
    out = parse_sentence(sent)
    if out is None:
        shown = None
    elif out[0] == 'fact':
        shown = ' & '.join(show_lit(x) for x in out[1])
    else:
        shown = ' ;; '.join(repr(r) for r in out[1])
    ok = out is not None and out[0] == kind and expect in shown
    print(('ok  ' if ok else 'FAIL'), sent)
    if not ok:
        print('   got', shown)
    assert ok, (sent, shown)

# "someone is red" by itself is not a fact I can store
assert parse_sentence("Someone is red.") is None
# and this is not a sentence I claim to understand
assert parse_sentence("Plungers suck.") is None
print('parser checks passed')

ok   Bob is young.
ok   The cat does not like the dog.
ok   Bob is red and kind.
ok   The bald eagle chases the cat.
ok   If someone is young, then they are smart.
ok   If someone is smart and kind, then they are successful.
ok   If someone is big and not rough then they are not green.
ok   All rabbits are furry.
ok   Kind, young things are not smart.
ok   Someone who is rough is quiet.
ok   If the cat is big then it is red.
ok   Bob is big if Bob is red.
ok   If something eats the bear and the bear eats the rabbit then it likes the cat.
parser checks passed


## 3. Reasoner

Forward chaining. Facts are indexed by `(negated, predicate)` so a body literal does not scan the whole set.

`_match` walks the body left to right. A positive literal has to unify with a known fact. A negative literal is ok when that negative fact was derived, or (closed world) when the positive one was not. If the variable is still free, as in "if something is not red", I try every object seen so far.

Rules that mention `not` in the body run after the plain rules in each round. Stop when a round adds nothing.

I keep the rule and the supporting facts on each derived literal so `explain` can print a trace. `steps` is how many unifications I tried.

In [10]:
# ---------- reasoning: forward chaining with unification ----------
def unify(pat, fact, theta):
    """pat and fact are literals of the same sign"""
    if pat[1] != fact[1] or len(pat[2]) != len(fact[2]):
        return None
    theta = dict(theta)
    for a, b in zip(pat[2], fact[2]):
        if is_var(a):
            if a in theta:
                if theta[a] != b: return None
            else:
                theta[a] = b
        elif a != b:
            return None
    return theta

def subst(l, theta):
    return (l[0], l[1], tuple(theta.get(a, a) for a in l[2]))

class KB:
    def __init__(self, facts, rules, closed_world=True):
        self.rules = rules
        self.closed_world = closed_world
        self.facts = set()
        self.index = defaultdict(list)       # (neg, pred) -> literals
        self.proof = {}                      # literal -> (rule or 'given', [support literals])
        self.steps = 0
        self.constants = set()
        for f in facts:
            self._add(f, 'given', [])
        for r in rules:
            for l in r.body + [r.head]:
                self.constants.update(a for a in l[2] if not is_var(a))

    def _add(self, l, how, support):
        if l in self.facts: return False
        # drop this if we already have the opposite.
        # "the cow visits the dog" must not be overwritten by not-visit(cow, dog).
        if (not l[0], l[1], l[2]) in self.facts: return False
        self.facts.add(l)
        self.index[(l[0], l[1])].append(l)
        self.proof[l] = (how, support)
        self.constants.update(a for a in l[2] if not is_var(a))
        return True

    def _match(self, body, theta, pos):
        """generator over bindings that satisfy body[pos:]"""
        if pos == len(body):
            yield theta
            return
        l = body[pos]
        if not l[0]:  # positive literal: unify against known facts
            for f in list(self.index[(False, l[1])]):
                self.steps += 1
                t = unify(l, f, theta)
                if t is not None:
                    yield from self._match(body, t, pos + 1)
        else:
            g = subst(l, theta)
            free = [a for a in g[2] if is_var(a)]
            if free:
                # nothing bound the variable yet (rule like 'if X is not red then X is big')
                # -> try every known object for it. closed world only.
                if not self.closed_world: return
                for c in list(self.constants):
                    t = dict(theta); t[free[0]] = c
                    yield from self._match(body, t, pos)
                return
            pos_version = (False, g[1], g[2])
            ok = g in self.facts or (self.closed_world and pos_version not in self.facts)
            if ok:
                yield from self._match(body, theta, pos + 1)

    def run(self, max_rounds=50):
        # negated-body rules go last so negation-as-failure sees the positive facts first
        ordered = sorted(self.rules, key=lambda r: any(b[0] for b in r.body))
        for _ in range(max_rounds):
            changed = False
            for r in ordered:
                for theta in list(self._match(r.body, {}, 0)):
                    h = subst(r.head, theta)
                    if any(is_var(a) for a in h[2]):
                        continue
                    support = [subst(b, theta) for b in r.body]
                    if self._add(h, r, support):
                        changed = True
            if not changed:
                break
        return self

    def entails(self, q):
        pos = (False, q[1], q[2])
        if q[0]:
            # a derived positive fact wins over a derived negation
            if pos in self.facts:
                return False
            return (True, q[1], q[2]) in self.facts or self.closed_world
        return pos in self.facts

    def explain(self, q, depth=0, seen=None):
        seen = seen or set()
        pad = '  ' * depth
        if q in self.proof:
            how, support = self.proof[q]
            if how == 'given':
                print(pad + show_lit(q) + '   [given]')
            else:
                print(pad + show_lit(q) + '   [by rule: ' + how.text.strip() + ']')
                for s in support:
                    if s in seen: continue
                    seen.add(s)
                    if s[0] and s not in self.proof:
                        print(pad + '  ' + show_lit(s) + '   [not derivable -> assumed false (closed world)]')
                    else:
                        self.explain(s, depth + 1, seen)
        elif q[0] and self.closed_world and (False, q[1], q[2]) not in self.facts:
            print(pad + show_lit(q) + '   [positive version not derivable -> true under closed world]')
        elif q[0]:
            print(pad + show_lit(q) + '   [FALSE: the positive fact is derivable]')
            self.explain((False, q[1], q[2]), depth + 1, seen)
        else:
            print(pad + show_lit(q) + '   [NOT derivable]')

In [11]:
# ---------- glue ----------
def predict(context, statement, closed_world=True, want_kb=False):
    facts, rules, failed = parse_context(context)
    q = parse_sentence(statement)
    if q is None or q[0] != 'fact':
        # could not read the query -> cheap fallback: exact string match
        ans = clean(statement) in [clean(x) for x in re.split(r'(?<=[.!?])\s+', context)]
        return (ans, None) if want_kb else ans
    kb = KB(facts, rules, closed_world).run()
    ans = all(kb.entails(l) for l in q[1])
    return (ans, kb) if want_kb else ans

In [12]:
# checks for the reasoner. these are the cases that bit me.

# unify: bind, then refuse a clash
assert unify(lit('young', '?x'), lit('young', 'bob'), {}) == {'?x': 'bob'}
assert unify(lit('chase', '?x', 'dog'), lit('chase', 'cat', 'dog'), {}) == {'?x': 'cat'}
assert unify(lit('young', '?x'), lit('kind', 'bob'), {}) is None
assert unify(lit('young', 'bob'), lit('young', 'anne'), {}) is None
th = unify(lit('like', '?x', '?y'), lit('like', 'cat', 'dog'), {})
assert unify(lit('like', '?x', 'mouse'), lit('like', 'cat', 'dog'), th) is None

# the example from the sheet
sheet = """Bob is young. Bob is kind. If someone is young, then they are smart.
If someone is smart and kind, then they are successful."""
assert predict(sheet, "Bob is successful.") is True
assert predict(sheet, "Bob is smart.") is True
assert predict(sheet, "Bob is tall.") is False
assert predict(sheet, "Bob is not successful.") is False

# closed world: never derived => the negation holds. open world: it does not.
assert predict("Anne is red.", "Anne is not blue.") is True
assert predict("Anne is red.", "Anne is blue.") is False
assert predict("Anne is red.", "Anne is not blue.", closed_world=False) is False

# a stated fact has to beat a rule that would conclude the opposite
cow = "The cow visits the dog. If something does not visit the cow then it does not visit the dog."
assert predict(cow, "The cow visits the dog.") is True
assert predict(cow, "The cow does not visit the dog.") is False

# negation has to wait until the positive chain has actually run
chain = """Bob is young. If someone is young then they are smart.
If someone is smart then they are red. If someone is not red then they are big."""
assert predict(chain, "Bob is red.") is True
assert predict(chain, "Bob is big.") is False
assert predict(chain, "Bob is not big.") is True

# "it" should be the cat, not some other object
assert predict("The cat is big. If the cat is big then it is red.", "The cat is red.") is True
assert predict("The cat chases the dog. If the cat chases the dog then it is big.", "The cat is big.") is True

# "not rough" in the middle of a rule, and a negative fact
nasty = """Fiona is white. If someone is big and not rough then they are not green.
If someone is white then they are not big."""
assert predict(nasty, "Fiona is not big.") is True
assert predict(nasty, "Fiona is green.") is False

print('reasoner checks passed')

reasoner checks passed


## 4. Sanity check - the example from the assignment sheet

In [13]:
ctx = """Bob is young. Bob is kind. If someone is young, then they are smart.
If someone is smart and kind, then they are successful."""

facts, rules, failed = parse_context(ctx)
print("facts:", [show_lit(f) for f in facts])
print("rules:")
for r in rules:
    print("  ", r)
print("unparsed:", failed)

for q in ["Bob is successful.", "Bob is not successful.", "Bob is smart.", "Bob is tall."]:
    ans, kb = predict(ctx, q, want_kb=True)
    print("\nQ:", q, "->", ans)
    kb.explain(parse_sentence(q)[1][0])

facts: ['young(bob)', 'kind(bob)']
rules:
   young(?x) -> smart(?x)
   smart(?x) & kind(?x) -> successful(?x)
unparsed: []

Q: Bob is successful. -> True
successful(bob)   [by rule: If someone is smart and kind, then they are successful.]
  smart(bob)   [by rule: If someone is young, then they are smart.]
    young(bob)   [given]
  kind(bob)   [given]

Q: Bob is not successful. -> False
not successful(bob)   [FALSE: the positive fact is derivable]
  successful(bob)   [by rule: If someone is smart and kind, then they are successful.]
    smart(bob)   [by rule: If someone is young, then they are smart.]
      young(bob)   [given]
    kind(bob)   [given]

Q: Bob is smart. -> True
smart(bob)   [by rule: If someone is young, then they are smart.]
  young(bob)   [given]

Q: Bob is tall. -> False
tall(bob)   [NOT derivable]


In [14]:
# a slightly nastier one with relations, negation and generic rules
ctx2 = """The dog is not cold. If someone is not red then they are big.
If something chases the cat and it is big then it visits the dog. The lion chases the cat.
The bear is kind. Kind people are nice. If the bear is nice then the bear chases the mouse.
Erin is a rabbit. All rabbits are furry. If someone visits the dog then they do not like the mouse.
Dave is rough. Someone who is rough is quiet. All things that are quiet are smart. Anne is red."""

for q in ["The lion visits the dog.", "The lion does not like the mouse.", "The bear chases the mouse.",
          "Erin is furry.", "The dog is cold.", "Dave is smart.", "Anne is big."]:
    print(q, "->", predict(ctx2, q))

print()
_, kb = predict(ctx2, "The lion visits the dog.", want_kb=True)
kb.explain(parse_sentence("The lion visits the dog.")[1][0])

The lion visits the dog. -> True
The lion does not like the mouse. -> True
The bear chases the mouse. -> True
Erin is furry. -> True
The dog is cold. -> False
Dave is smart. -> True
Anne is big. -> False

visit(lion, dog)   [by rule: If something chases the cat and it is big then it visits the dog.]
  chase(lion, cat)   [given]
  big(lion)   [by rule: If someone is not red then they are big.]
    not red(lion)   [not derivable -> assumed false (closed world)]


## 5. Public datasets

Column names differ, so I rename everything to `context`, `statement`, `flag`. Unknown / Uncertain becomes False: the assignment only wants entailed or not.

The loader tries a couple of Hugging Face ids. CounterBench's repo id fails because the extra json files have different columns, so I load the two data files by name.

In [15]:
CANDIDATES = {
    "ruletaker":    ["tasksource/ruletaker", "allenai/ruletaker", "tasksource/proofwriter", "theoxo/proofwriter-deduction-balanced"],
    "counterbench": ["CounterBench/CounterBench", "cbench/CounterBench"],
    "folio":        ["yale-nlp/FOLIO", "tasksource/folio"],
}

CTX_COLS  = ["context", "theory", "premises", "premise", "passage", "story", "kb", "given_info"]
STMT_COLS = ["statement", "question", "conclusion", "hypothesis", "query", "claim"]
LBL_COLS  = ["flag", "label", "answer", "gold_label", "target"]

TRUE_WORDS  = {"true", "1", "yes", "entailment", "entailed", "entails", "correct"}
FALSE_WORDS = {"false", "0", "no", "not entailment", "not_entailment", "contradiction",
               "unknown", "uncertain", "neutral", "incorrect", "neither", "none"}

def to_bool(v):
    if isinstance(v, bool): return v
    if isinstance(v, (int, float)): return bool(v)
    s = str(v).strip().lower()
    if s in TRUE_WORDS: return True
    if s in FALSE_WORDS: return False
    return None

def pick(cols, options):
    for o in options:
        if o in cols: return o
    return None

def normalise(ds, name):
    df = ds.to_pandas()
    c, s, l = pick(df.columns, CTX_COLS), pick(df.columns, STMT_COLS), pick(df.columns, LBL_COLS)
    if not (c and s and l):
        print(f"  [{name}] could not find columns in {list(df.columns)} - skipping")
        return None
    out = pd.DataFrame({"context": df[c].astype(str), "statement": df[s].astype(str)})
    out["flag"] = df[l].map(to_bool)
    before = len(out)
    out = out.dropna(subset=["flag"]).reset_index(drop=True)
    out["flag"] = out["flag"].astype(bool)
    out["source"] = name
    print(f"  [{name}] cols: {c}/{s}/{l}   rows: {len(out)} (dropped {before-len(out)} with unknown labels)")
    return out

def load_public(limit_per_set=None):
    frames = []
    for name, ids in CANDIDATES.items():
        got = None
        for hf_id in ids:
            try:
                dd = datasets.load_dataset(hf_id)
                split = "validation" if "validation" in dd else ("dev" if "dev" in dd else ("test" if "test" in dd else list(dd.keys())[0]))
                print(f"{name}: loaded {hf_id} (split={split})")
                got = normalise(dd[split], name)
                if got is not None: break
            except Exception as e:
                print(f"{name}: {hf_id} failed -> {type(e).__name__}: {str(e)[:80]}")
        # repo id fails (mixed json schemas), so read the two data files
        if got is None and name == "counterbench":
            parts = []
            for rel in ("data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"):
                url = "hf://datasets/CounterBench/CounterBench/" + rel
                try:
                    dd = datasets.load_dataset("json", data_files=url, split="train")
                    print(f"{name}: loaded {rel} ({len(dd)} rows)")
                    part = normalise(dd, name)
                    if part is not None:
                        parts.append(part)
                except Exception as e:
                    print(f"{name}: {rel} failed -> {type(e).__name__}: {str(e)[:80]}")
            if parts:
                got = pd.concat(parts, ignore_index=True)
        if got is not None:
            if limit_per_set: got = got.sample(min(limit_per_set, len(got)), random_state=0)
            frames.append(got)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["context","statement","flag","source"])

public = load_public(limit_per_set=2000)   # cap per dataset so the notebook runs in reasonable time; set None for everything
print(public.source.value_counts())
public.head()

README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

... (21 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

folio: loaded tasksource/folio (split=validation)
  [folio] cols: premises/conclusion/label   rows: 203 (dropped 0 with unknown labels)
source
ruletaker       2000
counterbench    1200
folio            203
Name: count, dtype: int64


,context,statement,flag,source
0,Dave is blue. Dave is nice. Dave is smart. Fio...,Dave is not big.,True,ruletaker
1,Gary is young. Harry is kind. Harry is nice. I...,Harry is not kind.,False,ruletaker
2,The bald eagle is green. The bald eagle is nic...,The lion sees the bald eagle.,True,ruletaker
3,Dave is kind. Dave is young. Fiona is green. A...,Fiona is green.,True,ruletaker
4,Charlie is red. If something is cold and rough...,Charlie is not rough.,True,ruletaker


### Evaluation helper

Reports accuracy, total / average inference time, how many statements the parser could read and how many unification steps were needed.

In [16]:
def evaluate(df, closed_world=True, label=""):
    t0 = time.time()
    preds, parsed_q, parsed_ctx, steps = [], 0, 0, 0
    for ctx, stmt in zip(df.context, df.statement):
        q = parse_sentence(stmt)
        if q is not None and q[0] == "fact":
            parsed_q += 1
        f, r, failed = parse_context(ctx)
        if not failed: parsed_ctx += 1
        ans, kb = predict(ctx, stmt, closed_world=closed_world, want_kb=True)
        if kb is not None: steps += kb.steps
        preds.append(ans)
    total = time.time() - t0
    acc = (pd.Series(preds, index=df.index) == df.flag).mean() * 100
    print(f"{label:28s} acc={acc:6.2f}%  n={len(df):5d}  time={total:7.2f}s  "
          f"({1000*total/max(len(df),1):.2f} ms/query)  parsed statements={100*parsed_q/max(len(df),1):5.1f}%  "
          f"fully parsed contexts={100*parsed_ctx/max(len(df),1):5.1f}%  unify steps={steps}")
    return pd.Series(preds, index=df.index), acc, total

### Experiment 1 - overall and per dataset (closed world)

In [17]:
results = {}
if len(public):
    preds_all, acc_all, t_all = evaluate(public, closed_world=True, label="ALL (closed world)")
    for name, part in public.groupby("source"):
        results[name] = evaluate(part, closed_world=True, label=name)

ALL (closed world)           acc= 80.22%  n= 3403  time=   1.66s  (0.49 ms/query)  parsed statements= 70.8%  fully parsed contexts= 91.1%  unify steps=109645
counterbench                 acc= 50.42%  n= 1200  time=   0.21s  (0.17 ms/query)  parsed statements= 20.8%  fully parsed contexts= 83.3%  unify steps=0
folio                        acc= 61.58%  n=  203  time=   0.07s  (0.34 ms/query)  parsed statements= 79.3%  fully parsed contexts= 49.3%  unify steps=37
ruletaker                    acc=100.00%  n= 2000  time=   1.38s  (0.69 ms/query)  parsed statements=100.0%  fully parsed contexts=100.0%  unify steps=109608


### Experiment 2 - closed world vs open world negation

Under the open-world setting a negated statement is only True if the negated fact is explicitly derived, and a rule with `not` in its body only fires if the negated fact is known. RuleTaker is defined with closed-world semantics so I expect that to win there; this is mostly to confirm the choice rather than to tune anything.

In [18]:
if len(public):
    for name, part in public.groupby("source"):
        evaluate(part, closed_world=False, label=name + " (open world)")

counterbench (open world)    acc= 50.42%  n= 1200  time=   0.21s  (0.18 ms/query)  parsed statements= 20.8%  fully parsed contexts= 83.3%  unify steps=0
folio (open world)           acc= 64.53%  n=  203  time=   0.07s  (0.34 ms/query)  parsed statements= 79.3%  fully parsed contexts= 49.3%  unify steps=37
ruletaker (open world)       acc= 75.25%  n= 2000  time=   1.33s  (0.66 ms/query)  parsed statements=100.0%  fully parsed contexts=100.0%  unify steps=98163


### Experiment 3 - where does it go wrong?

Breaking the errors down by whether the statement / context parsed. If most errors are on unparsed inputs, the reasoning is fine and the parser is the bottleneck.

In [19]:
if len(public):
    rows = []
    for i, (ctx, stmt, gold) in enumerate(zip(public.context, public.statement, public.flag)):
        q = parse_sentence(stmt)
        _, _, failed = parse_context(ctx)
        rows.append({"q_parsed": q is not None and q[0] == "fact",
                     "ctx_parsed": len(failed) == 0,
                     "correct": preds_all.iloc[i] == gold,
                     "source": public.source.iloc[i]})
    err = pd.DataFrame(rows)
    print(err.groupby(["source", "q_parsed", "ctx_parsed"]).correct.agg(["mean", "count"]).round(3))

    # a few sentences the parser rejected, useful for extending the patterns
    bag = []
    for ctx in public.context.sample(min(300, len(public)), random_state=1):
        bag += parse_context(ctx)[2]
    print("\nsample of unparsed sentences:")
    for s_ in random.Random(0).sample(bag, min(15, len(bag))): print("  -", s_)

                                   mean  count
source       q_parsed ctx_parsed              
counterbench False    False       0.525    200
                      True        0.500    750
             True     True        0.500    250
folio        False    False       0.583     24
                      True        0.556     18
             True     False       0.582     79
                      True        0.671     82
ruletaker    True     True        1.000   2000

sample of unparsed sentences:
  - Trune~Bern(0.7).
  - Skul~Bern(0.4).
  - Plungers suck.
  - Blaf~ Bern(0.2).
  - Plog~Bern(0.4).
  - People in this club who perform in school talent shows often attend and are very engaged with school events.
  - All of Kelly Wearstler's design styles that Max adores are evocative.
  - No brutalist buildings that Max adores have interesting geometries.
  - The monthly rent at the Olive Garden is $2000.
  - People in this club either perform in school talent shows often or are inactive and 

## 6. Reasoning traces on real examples

Picking a couple of RuleTaker items (one True, one False) and printing the derivation.

In [20]:
if len(public) and "ruletaker" in set(public.source):
    rt = public[public.source == "ruletaker"]
    shown = {True: 0, False: 0}
    for ctx, stmt, gold in zip(rt.context, rt.statement, rt.flag):
        q = parse_sentence(stmt)
        if q is None or q[0] != "fact" or shown[gold] >= 1: continue
        ans, kb = predict(ctx, stmt, want_kb=True)
        if ans != gold: continue
        shown[gold] += 1
        print("=" * 70)
        print("CONTEXT:", ctx)
        print("STATEMENT:", stmt, "| gold:", gold, "| predicted:", ans)
        print("derived", len(kb.facts), "facts in", kb.steps, "unification steps")
        print("TRACE:")
        kb.explain(q[1][0])
        if sum(shown.values()) == 2: break

CONTEXT: Dave is blue. Dave is nice. Dave is smart. Fiona is big. Fiona is blue. Fiona is furry. Fiona is kind. Fiona is nice. Gary is not furry. Gary is red. Harry is big. If someone is smart then they are kind. If Harry is smart then Harry is not kind. All red, big people are nice. If Fiona is furry and Fiona is blue then Fiona is nice. All red people are big. If someone is smart then they are blue. Blue, nice people are smart. Nice people are blue.
STATEMENT: Dave is not big. | gold: True | predicted: True
derived 18 facts in 139 unification steps
TRACE:
not big(dave)   [positive version not derivable -> true under closed world]
CONTEXT: Gary is young. Harry is kind. Harry is nice. If something is young then it is nice. If something is kind then it is big. Big things are quiet.
STATEMENT: Harry is not kind. | gold: False | predicted: False
derived 6 facts in 6 unification steps
TRACE:
not kind(harry)   [FALSE: the positive fact is derivable]
  kind(harry)   [given]


## 7. submission.csv

The private test file is expected to have an `id` column plus context / statement columns (names are detected the same way as above). Change `TEST_PATH` to wherever the file sits on Kaggle. If no test file is present the cell writes predictions for the public set instead so the output format can still be checked.

In [21]:
TEST_PATH = "/kaggle/input/test.csv"     # <- update

def read_any(path):
    if path.endswith(".csv"):   return pd.read_csv(path)
    if path.endswith(".json"):  return pd.read_json(path)
    if path.endswith(".jsonl"): return pd.read_json(path, lines=True)
    if path.endswith(".parquet"): return pd.read_parquet(path)
    raise ValueError("unknown file type: " + path)

if os.path.exists(TEST_PATH):
    test = read_any(TEST_PATH)
    c, s = pick(test.columns, CTX_COLS), pick(test.columns, STMT_COLS)
    idc = pick(test.columns, ["id", "ID", "Id", "idx", "index"])
    assert c and s, f"could not find context/statement columns in {list(test.columns)}"
    ids = test[idc] if idc else range(len(test))
    t0 = time.time()
    preds = [predict(str(ctx), str(stmt)) for ctx, stmt in zip(test[c], test[s])]
    t = time.time() - t0
    sub = pd.DataFrame({"id": ids, "flag": preds})
    print(f"predicted {len(sub)} rows in {t:.2f}s")
    if "flag" in test.columns:
        print("accuracy on this file:", (sub.flag.values == test.flag.map(to_bool).values).mean() * 100)
else:
    print("no test file found at", TEST_PATH, "- writing predictions for the public set instead")
    t0 = time.time()
    preds = [predict(ctx, stmt) for ctx, stmt in zip(public.context, public.statement)]
    t = time.time() - t0
    sub = pd.DataFrame({"id": range(len(public)), "flag": preds})
    if len(public):
        print(f"final accuracy: {(sub.flag.values == public.flag.values).mean()*100:.2f}%   total inference time: {t:.2f}s")

sub["flag"] = sub["flag"].map({True: "True", False: "False"})
sub.to_csv("submission.csv", index=False)
sub.head()

no test file found at /kaggle/input/test.csv - writing predictions for the public set instead
final accuracy: 80.22%   total inference time: 1.02s


,id,flag
0,0,True
1,1,False
2,2,True
3,3,True
4,4,True


In [22]:

# ============================================================
# ADDITIONAL EXPERIMENTS FOR REPORT V3
# 1. Majority and exact-match baselines
# 2. RuleTaker accuracy by inferred proof depth
# 3. Actual multi-step RuleTaker proof trace
# Requires: public, predict, parse_sentence, clean, KB
# ============================================================

import re
import pandas as pd
from collections import defaultdict

# ------------------------------------------------------------
# EXPERIMENT 4: BASELINES
# ------------------------------------------------------------

def majority_baseline(df):
    labels = df["flag"].astype(bool)
    return max(labels.mean(), 1 - labels.mean())

def string_match_baseline(df):
    predictions = []

    for ctx, stmt in zip(df.context, df.statement):
        sentences = re.split(r'(?<=[.!?])\s+', str(ctx))
        predictions.append(
            clean(str(stmt)) in [clean(s) for s in sentences]
        )

    return (
        pd.Series(predictions, index=df.index)
        == df.flag.astype(bool)
    ).mean()

print("\n=== BASELINE COMPARISON ===")

for name, part in public.groupby("source"):
    majority = majority_baseline(part)
    string_match = string_match_baseline(part)

    print(
        f"{name:15s} | "
        f"Majority: {majority*100:.2f}% | "
        f"String match: {string_match*100:.2f}%"
    )


# ------------------------------------------------------------
# EXPERIMENT 5: PROOF DEPTH
# Depth 0 = given fact
# Depth 1 = one rule application
# Depth 2 = two chained rule applications
# ------------------------------------------------------------

def literal_depth(lit, kb, visiting=None):
    if visiting is None:
        visiting = set()

    if lit in visiting:
        return None

    if lit not in kb.proof:
        return None

    how, supports = kb.proof[lit]

    if how == "given":
        return 0

    visiting = visiting | {lit}
    depths = []

    for support in supports:
        if support[0] and support not in kb.proof:
            depths.append(0)  # closed-world assumption
        else:
            d = literal_depth(support, kb, visiting)
            if d is not None:
                depths.append(d)

    return 1 + max(depths, default=0)


def get_query_depth(statement, kb):
    q = parse_sentence(statement)

    if q is None or q[0] != "fact":
        return "unparsed"

    depths = []

    for literal in q[1]:
        positive = (False, literal[1], literal[2])
        d = literal_depth(positive, kb)

        if d is not None:
            depths.append(d)

    if depths:
        return max(depths)

    if all(
        l[0] and
        (False, l[1], l[2]) not in kb.facts
        for l in q[1]
    ):
        return "CWA absence"

    return "no derivation"


rt = public[public.source == "ruletaker"]

depth_rows = []

for ctx, stmt, gold in zip(
    rt.context, rt.statement, rt.flag
):
    prediction, kb = predict(
        ctx, stmt, want_kb=True
    )

    depth = get_query_depth(stmt, kb)

    depth_rows.append({
        "depth": depth,
        "correct": prediction == gold
    })

depth_df = pd.DataFrame(depth_rows)

print("\n=== RULETAKER ACCURACY BY PROOF DEPTH ===")

print(
    depth_df.groupby("depth")["correct"]
    .agg(["count", "mean"])
    .assign(accuracy=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .to_string()
)


# ------------------------------------------------------------
# EXPERIMENT 6: REAL MULTI-STEP DATASET PROOF
# Find a gold-True positive query with depth >= 2
# ------------------------------------------------------------

print("\n=== MULTI-STEP RULETAKER PROOF ===")

found = False

for ctx, stmt, gold in zip(
    rt.context, rt.statement, rt.flag
):
    if not gold:
        continue

    q = parse_sentence(stmt)

    if q is None or q[0] != "fact":
        continue

    if any(lit[0] for lit in q[1]):
        continue

    prediction, kb = predict(
        ctx, stmt, want_kb=True
    )

    if not prediction:
        continue

    depths = [
        literal_depth(lit, kb)
        for lit in q[1]
    ]

    if depths and max(
        (d for d in depths if d is not None),
        default=0
    ) >= 2:

        print("CONTEXT:", ctx)
        print("QUERY:", stmt)
        print("GOLD:", gold)
        print("PREDICTION:", prediction)
        print("PROOF DEPTH:", max(depths))
        print("\nPROOF TREE:")

        for lit in q[1]:
            kb.explain(lit)

        found = True
        break

if not found:
    print("No multi-step positive proof found in this sample.")



=== BASELINE COMPARISON ===
counterbench    | Majority: 50.42% | String match: 50.42%
folio           | Majority: 64.53% | String match: 64.53%
ruletaker       | Majority: 51.15% | String match: 60.95%

=== RULETAKER ACCURACY BY PROOF DEPTH ===
               count  accuracy
depth                         
0                418     100.0
1                261     100.0
2                198     100.0
3                131     100.0
4                 31     100.0
5                 29     100.0
CWA absence      464     100.0
no derivation    468     100.0

=== MULTI-STEP RULETAKER PROOF ===
CONTEXT: Alan is young. Alan is big. Bob is nice. Bob is green. Bob is cold. Bob is blue. Eric is young. Eric is round. Eric is big. Harry is round. Harry is blue. If someone is blue and nice and green then they are round. If someone is green and kind and round then they are young. If someone is kind then they are red. If someone is blue and round then they are kind. If someone is red and nice and big the